In [ ]:
# --- Cell 1: Import Libraries ---
import tensorflow as tf
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("Tất cả thư viện đã được import thành công!")

Tất cả thư viện đã được import thành công!


In [ ]:
# --- Cell 2: Setup Data Paths ---
# Đường dẫn gốc của dự án
base_dir = r'E:\btlhttm\BTL-HTTM-2025'

# Đường dẫn đến thư mục dữ liệu cats_vs_dogs_small
data_dir = os.path.join(base_dir, 'cats_vs_dogs_small')
train_dir = os.path.join(data_dir, 'train')
validation_dir = os.path.join(data_dir, 'validation')

print(f"Thư mục training: {train_dir}")
print(f"Thư mục validation: {validation_dir}")

Thư mục training: E:\btlhttm\BTL-HTTM-2025\cats_vs_dogs_small\train
Thư mục validation: E:\btlhttm\BTL-HTTM-2025\cats_vs_dogs_small\validation


In [9]:
# --- Cell 3 (New Version): Create tf.data.Dataset ---

# Thiết lập các tham số chung
image_size = (150, 150)
batch_size = 32

# Tạo bộ dữ liệu training từ thư mục
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='binary',
    image_size=image_size,
    batch_size=batch_size,
    shuffle=True # Xáo trộn dữ liệu training là một best practice
)

# Tạo bộ dữ liệu validation từ thư mục
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    labels='inferred',
    label_mode='binary',
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False # Không cần xáo trộn dữ liệu validation
)

print("Đã tạo xong train_dataset và validation_dataset.")

Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Đã tạo xong train_dataset và validation_dataset.


In [10]:
# --- Cell 4 (Updated): Add Augmentation Layers to the Model ---
from tensorflow.keras import layers

# Tạo một "mô hình con" chỉ chứa các lớp augmentation
data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

# Xây dựng mô hình chính, với lớp augmentation là lớp đầu tiên
model = Sequential([
    # Input layer và rescale
    layers.Input(shape=(150, 150, 3)),
    layers.Rescaling(1./255),
    
    # 1. Lớp Data Augmentation
    data_augmentation,
    
    # 2. Các lớp CNN còn lại
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)                │ (None, 150, 150, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ data_augmentation (Sequential)       │ (None, 150, 150, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 36, 36, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 34, 34, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 17, 17, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 512)                 │      18,940,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │             513 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,034,177 (72.61 MB)

 Trainable params: 19,034,177 (72.61 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# --- Cell 5: Define "Best Practices" Callbacks ---

# 1. EarlyStopping: Dừng sớm nếu không có cải thiện
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    verbose=1,
    restore_best_weights=True
)

# 2. ModelCheckpoint: Lưu lại phiên bản model tốt nhất
model_checkpoint = ModelCheckpoint(
    filepath='best_model_from_hung_kim.keras', # Tên file chứa model tốt nhất
    save_best_only=True,
    monitor='val_loss',
    verbose=1
)

# Tạo một danh sách chứa các callbacks sẽ sử dụng
callbacks_list = [early_stopping, model_checkpoint]

print("Callbacks 'EarlyStopping' và 'ModelCheckpoint' đã sẵn sàng!")

Callbacks 'EarlyStopping' và 'ModelCheckpoint' đã sẵn sàng!


In [11]:
# --- Cell 6 (Updated): Train the Model with tf.data.Dataset ---
history = model.fit(
    train_dataset,
    epochs=100,
    validation_data=validation_dataset,
    callbacks=callbacks_list # Callbacks vẫn giữ nguyên
)

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 439ms/step - accuracy: 0.4938 - loss: 0.9823
Epoch 1: val_loss did not improve from 0.47614
63/63 ━━━━━━━━━━━━━━━━━━━━ 40s 493ms/step - accuracy: 0.5095 - loss: 0.7729 - val_accuracy: 0.5150 - val_loss: 0.6919
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 395ms/step - accuracy: 0.5232 - loss: 0.6913
Epoch 2: val_loss did not improve from 0.47614
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 433ms/step - accuracy: 0.5455 - loss: 0.6888 - val_accuracy: 0.6140 - val_loss: 0.6705
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 395ms/step - accuracy: 0.5823 - loss: 0.6713
Epoch 3: val_loss did not improve from 0.47614
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 434ms/step - accuracy: 0.5980 - loss: 0.6651 - val_accuracy: 0.5930 - val_loss: 0.6740
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 390ms/step - accuracy: 0.6257 - loss: 0.6551
Epoch 4: val_loss did not improve from 0.47614
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 428ms/step - accuracy: 0.6370 - loss: 0.6424 - val_accuracy: 0.6540 - val_loss:

In [7]:
# --- Cell 7: Load and Verify the Best Model ---
print("\n--- Huấn luyện đã hoàn tất ---")

# Tải lại mô hình tốt nhất đã được lưu
best_model = tf.keras.models.load_model('best_model_from_hung_kim.keras')

print("\nĐã tải xong mô hình tốt nhất.")
print("Đây là mô hình nên được sử dụng để đánh giá trên tập test.")

# In ra cấu trúc của mô hình tốt nhất để xác nhận
best_model.summary()


--- Huấn luyện đã hoàn tất ---

Đã tải xong mô hình tốt nhất.
Đây là mô hình nên được sử dụng để đánh giá trên tập test.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 36, 36, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 34, 34, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 17, 17, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │      18,940,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             513 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 57,102,533 (217.83 MB)

 Trainable params: 19,034,177 (72.61 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 38,068,356 (145.22 MB)

In [8]:
# --- Cell 8: Evaluate the Best Model on the Test Set ---

# Đường dẫn đến thư mục test
test_dir = os.path.join(data_dir, 'test')
print(f"Thư mục test: {test_dir}")

# Tạo generator cho dữ liệu test (CHỈ RESCALE, KHÔNG AUGMENT)
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    shuffle=False # Không cần xáo trộn dữ liệu test
)

# Dùng mô hình tốt nhất để đánh giá
print("\nĐang tiến hành đánh giá mô hình trên tập Test...")
test_loss, test_accuracy = best_model.evaluate(test_generator)

print(f"\nKết quả cuối cùng trên tập Test:")
print(f"   - Mất mát (Loss): {test_loss:.4f}")
print(f"   - Độ chính xác (Accuracy): {test_accuracy*100:.2f}%")

Thư mục test: E:\btlhttm\BTL-HTTM-2025\cats_vs_dogs_small\test
Found 2000 images belonging to 2 classes.

Đang tiến hành đánh giá mô hình trên tập Test...
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 163ms/step - accuracy: 0.7515 - loss: 0.5034

Kết quả cuối cùng trên tập Test:
   - Mất mát (Loss): 0.5034
   - Độ chính xác (Accuracy): 75.15%
